# Costs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/04_costs.ipynb)

Cost functions as composable objects: the running and terminal costs that turn a trajectory into the objective $J$.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. A cost is a running cost and a terminal cost

A `CostFunction` turns a trajectory into a number. It holds two equations with the model's
signature, the running cost $g(x, u, t)$ and the terminal cost $h(x, t)$, and the
objective of a run on $[0, t_f]$ is

$$J = \int_0^{t_f} e^{-\rho t}\, g(x, u, t)\, dt + e^{-\rho t_f}\, h(x(t_f), t_f).$$

The discount rate $\rho$ and the horizon $t_f$ belong to the problem, not to the cost, so
one cost serves many problems. `QuadraticCost.from_system` builds
$g = (x - \bar x)^\top Q (x - \bar x) + (u - \bar u)^\top R (u - \bar u)$ with the plant's
dimensions; here the target $\bar x$ is the pendulum upright.

In [ ]:
import numpy as np

from minilink import Pendulum, QuadraticCost

plant = Pendulum()
UP = np.array([np.pi, 0.0])

cost = QuadraticCost.from_system(
    plant, Q=np.diag([1.0, 0.1]), R=0.01 * np.eye(1), xbar=UP
)
cost

`integrate` evaluates the running cost along a `Trajectory`: it returns the integrand at
each sample and its running integral by the trapezoid rule. Adding the terminal cost gives
$J$. Here is the price of letting the pendulum fall from near the top with no torque.

In [ ]:
plant.x0 = np.array([3.0, 0.0])
traj = plant.compute_trajectory(tf=5.0)

cost_rate, running_cost = cost.integrate(traj)
J = running_cost[-1] + cost.h(traj.x[:, -1], traj.t[-1])
J

## 2. Costs add and scale

`a + b` is a `SumCost` and `w * a` a `ScaledCost`. Fields and sets bring their own costs
(see `03_sets_fields_distributions`): `V.as_terminal_cost()` charges $h = V(x)$ at the end,
and `X.as_cost(weight=w)` is the soft form of a state constraint, zero inside and a
quadratic hinge outside. A sum keys its terms by id, as a diagram keys its blocks, and its
`params` is the nested view of the terms' numbers.

In [ ]:
from minilink.core import QuadraticField

V = QuadraticField(UP, np.diag([10.0, 1.0]))
X = plant.state.box

total = cost + 5.0 * V.as_terminal_cost() + X.as_cost(weight=100.0)
print(total)
list(total.params)

In [ ]:
x = np.array([3.0, 0.0])
u = np.array([1.0])

total.g(x, u), total.h(x)

A cost has no `with_params`; its stored numbers change by assignment. On a sum, a partial
dictionary changes only the terms it names, and the term's own `params` follows.

In [ ]:
total.params = {"QuadraticCost": {**cost.params, "R": np.eye(1)}}
cost.params["R"]

## 3. Minimum time

`TimeCost` is $g = 1$ everywhere except within `eps` of the target, where it is $0$: the
objective is the time to reach the target, and its optimal policy is bang-bang. Its target
and radius are its parameters.

In [ ]:
from minilink import TimeCost

time_cost = TimeCost.from_system(plant, xbar=UP, eps=0.1)
time_cost.g(x, u), time_cost.g(UP, u), time_cost.params

## 4. A custom cost

A custom cost subclasses `CostFunction` and writes `g` and `h`, with the same conventions as
a model's `f`: numbers in `params`, read through the override, and arrays built with
`array_module` so the cost traces under JAX. The swing-up cost below wraps the angle error,
so the bottom position sees the upright target on both sides.

In [ ]:
from minilink import CostFunction
from minilink.core.backends import array_module


class SwingUpCost(CostFunction):
    def __init__(self):
        self.params = {"w_dtheta": 0.1, "w_u": 0.001}

    def g(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        w_dtheta = params["w_dtheta"]
        w_u = params["w_u"]

        theta = x[0]
        dtheta = x[1]
        tau = u[0]

        # e = θ − π wrapped to (−π, π]
        e = xp.arctan2(xp.sin(theta - xp.pi), xp.cos(theta - xp.pi))
        g = e**2 + w_dtheta * dtheta**2 + w_u * tau**2

        return g

    def h(self, x, t=0.0, params=None):
        return 0.0

In [ ]:
import jax.numpy as jnp

swing_up = SwingUpCost()
(
    swing_up.g(np.array([-3.1, 0.0]), u),
    swing_up.g(jnp.array([3.1, 0.0]), jnp.array([1.0])),
)

## 5. Costs on a clock: `StepCost` and `SampledCost`

A step problem pays a stage cost per tick: a `StepCost` holds $c(x, u, k)$ and $h(x, k)$,
with $J = \sum_{k<N} \gamma^k c(x_k, u_k, k) + \gamma^N h(x_N, N)$. `SampledCost(cost,
plant, dt)` is the stage cost of a continuous cost on a sampled plant:
$c_k = \int_0^{\Delta t} e^{-\rho \tau} g\, d\tau$ along the plant's motion over one period,
the input held. Under one Euler step it is exactly $g(x_k, u_k, t_k)\,\Delta t$. A compiled
problem builds it for you (see `05_problem`); a native step model writes its own `StepCost`.

In [ ]:
from minilink.core import SampledCost

stage_cost = SampledCost(cost, plant, 0.05)
stage_cost_euler = SampledCost(cost, plant, 0.05, solver="euler")

stage_cost.c(x, u, 0), stage_cost_euler.c(x, u, 0), cost.g(x, u) * 0.05

## Summary

| Object | Equation | Built by |
| --- | --- | --- |
| `QuadraticCost` | $g = \lVert x - \bar x\rVert_Q^2 + \lVert u - \bar u\rVert_R^2$, $h = \lVert x - \bar x\rVert_S^2$ | `QuadraticCost.from_system(sys, Q=, R=, xbar=)` |
| `TimeCost` | $g = 1$ off the target | `TimeCost.from_system(sys, xbar=, eps=)` |
| `SumCost`, `ScaledCost` | $g = \sum_i w_i g_i$ | `a + b`, `w * a` |
| field and set costs | $g = w\, V(x)$, $h = w\, V(x)$, hinge on the margin | `V.as_cost`, `V.as_terminal_cost`, `X.as_cost` |
| custom | your $g$ and $h$ | subclass `CostFunction` |
| `StepCost`, `SampledCost` | $c(x, u, k)$ per tick | subclass, or `SampledCost(cost, plant, dt)` |

Next, [`05_problem`](05_problem.ipynb): a system, its sets, a cost and a horizon make a
`PlanningProblem`.